# 03 - Graph Attention Network Training

Train the AirspaceGATv2 graph attention network for multi-aircraft
interaction modeling. Aircraft are nodes with trajectory embeddings
as features, connected by proximity and altitude overlap edges.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from aeroconform.config import AeroConformConfig

config = AeroConformConfig()
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"Graph config: hidden={config.graph_hidden}, heads={config.graph_heads}, "
      f"layers={config.graph_layers}, edge_dim={config.edge_dim}")

## Create Foundation Model and Generate Embeddings

We need a trained (or at least initialized) foundation model to produce
trajectory embeddings that serve as node features for the graph.

In [ ]:
from aeroconform.models.trajectory_model import TrajectoryTransformer

foundation_model = TrajectoryTransformer.from_config(config).to(device)
foundation_model.eval()

print(f"Foundation model loaded: {sum(p.numel() for p in foundation_model.parameters()):,} params")

## Build a Synthetic Airspace Snapshot

Generate a snapshot of aircraft in the LIMM FIR and build the
airspace graph with proximity-based edges.

In [ ]:
from aeroconform.data.graph_builder import AirspaceGraphBuilder

rng = np.random.default_rng(42)
n_aircraft = 15

# Create synthetic aircraft states in the LIMM FIR
states_data = {
    "icao24": [f"ac{i:04d}" for i in range(n_aircraft)],
    "latitude": rng.uniform(config.bbox[0], config.bbox[1], n_aircraft),
    "longitude": rng.uniform(config.bbox[2], config.bbox[3], n_aircraft),
    "baro_altitude": rng.uniform(15000, 40000, n_aircraft),
    "velocity": rng.uniform(200, 500, n_aircraft),
    "true_track": rng.uniform(0, 360, n_aircraft),
    "vertical_rate": rng.normal(0, 100, n_aircraft),
}
states_df = pd.DataFrame(states_data)
print("Aircraft states:")
print(states_df.head(10))

# Generate trajectory embeddings from the foundation model
# Create synthetic trajectory inputs for each aircraft
seq_len = config.seq_len
input_dim = config.input_dim
x_batch = torch.randn(n_aircraft, seq_len, input_dim).to(device)

with torch.no_grad():
    embeddings = foundation_model.get_trajectory_embedding(x_batch)  # (N, d_model)

print(f"\nEmbeddings shape: {embeddings.shape}")

# Build the airspace graph
builder = AirspaceGraphBuilder(config=config)
graph = builder.build_graph(states_df, embeddings.cpu())

print(f"\nGraph:")
print(f"  Nodes (x): {graph['x'].shape}")
print(f"  Edges (edge_index): {graph['edge_index'].shape}")
print(f"  Edge features (edge_attr): {graph['edge_attr'].shape}")

## Visualize the Airspace Graph

Plot the spatial layout of aircraft with edges connecting proximate pairs.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

lats = states_df["latitude"].values
lons = states_df["longitude"].values

# Draw edges
edge_index = graph["edge_index"].numpy()
for e in range(edge_index.shape[1]):
    i, j = edge_index[0, e], edge_index[1, e]
    ax.plot([lons[i], lons[j]], [lats[i], lats[j]],
            "gray", alpha=0.3, linewidth=0.8)

# Draw nodes
scatter = ax.scatter(
    lons, lats,
    c=states_df["baro_altitude"].values,
    cmap="viridis", s=100, zorder=5, edgecolors="black", linewidth=0.5,
)
plt.colorbar(scatter, ax=ax, label="Altitude (ft)")

# Label aircraft
for i, icao in enumerate(states_df["icao24"]):
    ax.annotate(icao, (lons[i], lats[i]), textcoords="offset points",
                xytext=(5, 5), fontsize=7)

# Draw bounding box
bbox = config.bbox
ax.plot(
    [bbox[2], bbox[3], bbox[3], bbox[2], bbox[2]],
    [bbox[0], bbox[0], bbox[1], bbox[1], bbox[0]],
    "r--", linewidth=1.5, label="LIMM FIR",
)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(f"Airspace Graph: {n_aircraft} aircraft, {edge_index.shape[1]} edges")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Instantiate AirspaceGATv2

The GATv2 model uses multi-head attention over aircraft pairs with
edge features (distance, closing speed, altitude difference, bearing).

In [ ]:
from aeroconform.models.graph_attention import AirspaceGATv2

gat_model = AirspaceGATv2.from_config(config).to(device)
n_params = sum(p.numel() for p in gat_model.parameters())

print(f"AirspaceGATv2 parameters: {n_params:,}")
print(f"Architecture:")
print(f"  in_channels:  {config.d_model} (from foundation model)")
print(f"  hidden:       {config.graph_hidden}")
print(f"  out_channels: {config.d_model}")
print(f"  heads:        {config.graph_heads}")
print(f"  layers:       {config.graph_layers}")
print(f"  edge_dim:     {config.edge_dim}")

## Forward Pass through GATv2

Run the graph data through the model and inspect the context-enriched
node embeddings and attention weights.

In [ ]:
gat_model.eval()

x = graph["x"].to(device)
edge_index = graph["edge_index"].to(device)
edge_attr = graph["edge_attr"].to(device)

with torch.no_grad():
    enriched, attention_weights = gat_model(x, edge_index, edge_attr)

print(f"Input node features:    {x.shape}")
print(f"Enriched node features: {enriched.shape}")
print(f"Number of GATv2 layers: {len(attention_weights)}")

for i, (edge_idx, alphas) in enumerate(attention_weights):
    print(f"\nLayer {i}:")
    print(f"  Edge index shape:     {edge_idx.shape}")
    print(f"  Attention weights:    {alphas.shape}")
    print(f"  Attention mean:       {alphas.mean().item():.6f}")
    print(f"  Attention min/max:    [{alphas.min().item():.6f}, {alphas.max().item():.6f}]")

## Visualize Attention Weights

Build an attention matrix from the edge-level attention coefficients
and display as a heatmap.

In [ ]:
# Build attention matrix from last layer's weights
last_edge_idx, last_alphas = attention_weights[-1]
last_edge_idx = last_edge_idx.cpu().numpy()
last_alphas = last_alphas.cpu().numpy()

# Average over heads if multi-head
if last_alphas.ndim > 1:
    last_alphas = last_alphas.mean(axis=-1)

attn_matrix = np.zeros((n_aircraft, n_aircraft))
for e in range(last_edge_idx.shape[1]):
    src, dst = last_edge_idx[0, e], last_edge_idx[1, e]
    attn_matrix[src, dst] = last_alphas[e]

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(attn_matrix, cmap="viridis", aspect="auto")
plt.colorbar(im, ax=ax, label="Attention Weight")

labels = states_df["icao24"].tolist()
ax.set_xticks(range(n_aircraft))
ax.set_yticks(range(n_aircraft))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(labels, fontsize=8)

ax.set_title("GATv2 Attention Weights (Last Layer)")
ax.set_xlabel("Target Aircraft")
ax.set_ylabel("Source Aircraft")
plt.tight_layout()
plt.show()

## Variable-Size Graph Support

Verify that AirspaceGATv2 handles graphs of different sizes
without any modifications.

In [ ]:
print("Testing variable-size graph support:")

for n_nodes in [3, 10, 25, 50]:
    x_test = torch.randn(n_nodes, config.d_model).to(device)
    # Create random edges
    n_edges = min(n_nodes * 2, n_nodes * (n_nodes - 1) // 2)
    if n_edges > 0:
        src = torch.randint(0, n_nodes, (n_edges,))
        dst = torch.randint(0, n_nodes, (n_edges,))
        ei_test = torch.stack([src, dst]).to(device)
        ea_test = torch.randn(n_edges, config.edge_dim).to(device)
    else:
        ei_test = torch.zeros((2, 0), dtype=torch.long).to(device)
        ea_test = torch.zeros((0, config.edge_dim)).to(device)

    with torch.no_grad():
        out, attn = gat_model(x_test, ei_test, ea_test)

    print(f"  N={n_nodes:3d}, E={n_edges:3d} -> output shape: {out.shape}  [OK]")

print("\nAll variable-size tests passed.")

## Spatial Attention Visualization

Overlay attention weights on the spatial graph to see which
aircraft interactions the model considers most important.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

# Draw edges with attention-weight-based coloring and thickness
original_ei = graph["edge_index"].numpy()
for e in range(last_edge_idx.shape[1]):
    src_idx, dst_idx = last_edge_idx[0, e], last_edge_idx[1, e]
    weight = last_alphas[e]
    ax.plot(
        [lons[src_idx], lons[dst_idx]],
        [lats[src_idx], lats[dst_idx]],
        color=plt.cm.Reds(weight / max(last_alphas.max(), 1e-8)),
        alpha=0.5 + 0.5 * weight / max(last_alphas.max(), 1e-8),
        linewidth=1 + 3 * weight / max(last_alphas.max(), 1e-8),
    )

# Draw nodes
ax.scatter(lons, lats, c="steelblue", s=100, zorder=5,
           edgecolors="black", linewidth=0.5)
for i, icao in enumerate(labels):
    ax.annotate(icao, (lons[i], lats[i]), textcoords="offset points",
                xytext=(5, 5), fontsize=7)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("GATv2 Attention on Airspace Graph (edge color/width = attention)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated:

1. **Foundation model embeddings** as node features for the graph
2. **AirspaceGraphBuilder** constructing proximity-based graphs
3. **AirspaceGATv2** forward pass with edge features
4. **Attention weight visualization** as heatmap and spatial overlay
5. **Variable-size graph support** verified for N=3..50

Next: `04_conformal_calibration.ipynb` calibrates the conformal prediction layer.